The model card template makes use of Jinja, hence we need to install the necessary package.

In [1]:
!pip install Jinja2

Required import statement

In [2]:
from huggingface_hub import ModelCard, ModelCardData

/Users/jackrong/University/34812-cwk-S-Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Before running the cell below, upload the model card template (`COMP34812_modelcard_template.md`) provided to you using the Colab file browser (on the left-hand side).

In [3]:
card_data = ModelCardData(
    language='en',
    license='cc-by-4.0',
    tags=['text-classification'],
    repo="https://github.com/jackr03/34812-cwk-S-Project",
    ignore_metadata_errors=True)

card = ModelCard.from_template(
    card_data = card_data,
    template_path='COMP34812_modelcard_template.md',
    model_id = 'x06213zr-j84846ky-NLI',

    model_summary = '''This is a binary Natural Language Inference (NLI) classification model that, given a premise and hypothesis, predicts whether the premise entails the hypothesis.
    It is based on the `Qwen/Qwen3.5-9B-Base` causal language model fine-tuned via supervised instruction tuning with LoRA (Low-Rank Adaptation) adapters.''',

    model_description = '''This model is `Qwen/Qwen3.5-9B-Base` (~9B parameters) fine-tuned for binary NLI using parameter-efficient fine-tuning (PEFT) with LoRA adapters via the
    Hugging Face TRL `SFTTrainer`. Premise-hypothesis pairs are formatted as structured instruction-tuning prompts using Qwen3's chat template, and the model is trained to generate
    the correct label token (`"0"` or `"1"`) as a single-token completion. LoRA adapters are applied to 7 projection layers
    (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`), keeping the base model weights frozen.

    At inference time, a `_LabelOnlyLogitsProcessor` masks all vocabulary tokens except `"0"` and `"1"` to `-inf`, ensuring the model always outputs a valid binary label.

    The prompt format (using Qwen3's chat template with `enable_thinking=False`) is:
    - **System:** "You are a binary Natural Language Inference classifier. Given a premise and a hypothesis, predict whether the premise entails the hypothesis."
    - **User:** Premise: {premise} / Hypothesis: {hypothesis} / Return ONLY the number of the correct label. / 0 = Non-entailment / 1 = Entailment / Answer:
    - **Assistant (training target):** "0" or "1" (single token)''',

    developers = 'Zhijie Rong and Kyan Yip',
    base_model_repo = 'https://huggingface.co/Qwen/Qwen3.5-9B-Base',
    base_model_paper = 'https://qwen.ai/blog?id=qwen3.5',
    model_type = 'Fine-tuned (supervised instruction tuning)',
    model_architecture = 'Transformer (decoder-only, Qwen3.5)',
    language = 'English',
    base_model = 'Qwen/Qwen3.5-9B-Base (~9B parameters)',

    training_data = '''24,432 premise-hypothesis pairs supplied by the COMP34812 coursework dataset (train split).
    Each example is converted to the full chat-template conversation (system + user + assistant turns).
    The prompt tokens are masked with `-100` in the loss computation so that only the label token contributes to training loss.
    Sequences are padded/truncated to 256 tokens.''',

    hyperparameters = '''
      - seed: 100
      - num_epochs: 1
      - per_device_train_batch_size: 16
      - gradient_accumulation_steps: 4  (effective batch size: 64)
      - learning_rate: 1e-4
      - lr_scheduler: cosine
      - max_seq_length: 256 tokens
      - optimizer: AdamW (default)
      - precision: bfloat16
      - LoRA rank (r): 16
      - LoRA alpha: 32
      - LoRA dropout: 0.0
      - LoRA target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj''',

    speeds_sizes_times = '''
      - overall training time: ~57 minutes (3,433 s)
      - LoRA adapter size: ~135 MB
      - base model size: ~18 GB (bfloat16)''',

    testing_data = '''6,736 premise-hypothesis pairs from the dev split of the COMP34812 coursework dataset.
    Test labels are held out (3,302 samples). The dev set is approximately balanced: class 0 — 3,258 samples; class 1 — 3,478 samples (~48.4% / 51.6%).''',

    testing_metrics = '''
      - Accuracy
      - Macro Precision
      - Macro Recall
      - Macro F1
      - Weighted Precision
      - Weighted Recall
      - Weighted F1
      - Per-class Precision, Recall, and F1 (class 0: Non-entailment; class 1: Entailment)
      ''',

    results = '''
    Given a zero-shot baseline of 67.83% accuracy (macro F1: 0.6501) on the unmodified `Qwen3.5-9B-Base`, fine-tuning with LoRA for 1 epoch achieved:
      - Accuracy:               93.29%  (+25.46 percentage points over baseline)
      - Weighted Precision:     0.9330
      - Weighted Recall:        0.9329
      - Weighted F1:            0.9329
      - Macro Precision:        0.9330
      - Macro Recall:           0.9330
      - Macro F1:               0.9330

    Per-class breakdown (fine-tuned model):
      - Class 0 (Non-entailment): Precision 0.924 | Recall 0.938 | F1 0.931 | Support 3,258
      - Class 1 (Entailment):     Precision 0.941 | Recall 0.928 | F1 0.935 | Support 3,478

    Performance is balanced across both classes (F1 difference of 0.004).
    ''',

    hardware_requirements = '''
      - GPU: NVIDIA A100-SXM4-80GB (79.25 GB VRAM required for full fine-tuning in bf16)
      - RAM: at least 40 GB system RAM
      - Storage: at least 25 GB (base model ~18 GB + adapter + dataset)''',

    software = '''
      - PyTorch >=2.0
      - Hugging Face Transformers
      - PEFT (Parameter-Efficient Fine-Tuning)
      - TRL (SFTTrainer)
      - Optuna (optional, for hyperparameter search)
      - pandas, scikit-learn''',

    bias_risks_limitations = '''
    1. Inputs (concatenation of premise + hypothesis + prompt overhead) longer than 256 tokens are truncated, which may degrade accuracy on longer inputs.
    2. The model was fine-tuned exclusively on English text and will not work reliably with other languages.
    3. Training data is derived from academic NLI corpora (SNLI/MultiNLI style), so performance may degrade on informal, conversational, or domain-specific text (e.g., legal, medical).
    4. The model is sensitive to the exact prompt format used during training; deviations from the expected system/user structure may affect predictions.
    5. The base model (`Qwen3.5-9B-Base`) was pre-trained on a large web corpus; biases present in that pre-training data may carry over into the fine-tuned model.''',

    additional_information = '''Hyperparameters (learning rate, LoRA rank, LoRA alpha, dropout) can be selected via an optional Optuna sweep consisting of 8 trials.
    The reported results use 1 training epoch; extended training with learning rate tuning may yield further improvements.
    Reported metrics are from the dev split only; generalisation to the held-out test set and out-of-distribution data is not confirmed.'''
)

# the following lines will write a markdown (.md) file; this becomes one of your model cards
# change the filename accordingly
with open('../C_model_card.md', 'w') as model_card:
  model_card.write(card.content)

Repo card metadata block was not found. Setting CardData to empty.
